In [13]:
from Utils.import_packages import *


# Methodology

Sentiment labels are assigned to each news article based on the sign of this aggregated `three-day excess return`. This excess return is calculated from the day a news article is first published and extends over the two subsequent days. To elaborate, excess return is defined as the difference between the return of a particular stock and the overall market return on the same day. This calculation is not limited to the day the news is published; instead, it aggregates the returns for the following two days as well, providing a
comprehensive three-day outlook.

A positive aggregated excess return leads to a sentiment label of `1`, indicating a positive sentiment. Conversely, a non-positive aggregated excess return results in a sentiment label of `0`, suggesting a negative sentiment.

In [14]:
stocks = pd.read_csv("Data/stocks_data.csv", header=[0, 1], index_col=0)
stocks_info = pd.read_csv("Data/SPY_companies_info.csv")
# Separate DataFrames by the top-level column (first row of headers)
stocks_dict = {key: stocks[key] for key in stocks.columns.levels[0]}
market = pd.read_csv("Data/market_data.csv")
market = market.set_index('Date')

In [15]:
# Calculate excess return sentiment label
def excess_return_sentiment_label(stock_data, market_data, num_days=3, price_used="Adj Close"):
    if price_used not in stock_data.columns or price_used not in market_data.columns:
        raise ValueError(f"Column '{price_used}' is missing in stock or market data.")

    stock_df = stock_data[[price_used]].copy()
    market_df = market_data[[price_used]].copy()
    market_df = market_df.add_prefix("SPY_")
    merged_df = stock_df.join(market_df)
    merged_df = merged_df.pct_change()


    merged_df["excess_returns"] = merged_df["Adj Close"] - merged_df["SPY_Adj Close"]
    merged_df["X_days_excess_returns"] =\
        (
            (1 + merged_df["excess_returns"])
            .rolling(num_days)
            .apply(lambda x: x.cumprod()[-1], raw=True)
            .shift(-(num_days - 1))
            - 1
        )
    # Label for 1 excess returns > 0, 0 otherwise
    merged_df["sentiment_label"] = (merged_df["X_days_excess_returns"] > 0).astype(int)
    merged_df = merged_df.reset_index()
    merged_df = merged_df[["Date", "sentiment_label"]]
    return merged_df


ticker_senti_date = pd.DataFrame()

for ticker in stocks_dict:
    result = excess_return_sentiment_label(stocks_dict[ticker], market)
    result["Ticker"] = ticker
    ticker_senti_date = pd.concat([ticker_senti_date, result], ignore_index=True)

# ticker_senti_date.dropna(subset=["sentiment_label"])
ticker_senti_date

,Date,sentiment_label,Ticker
0,2014-01-02,0,A
1,2014-01-03,1,A
2,2014-01-06,1,A
3,2014-01-07,1,A
4,2014-01-08,1,A
...,...,...,...
1386766,2024-12-09,1,ZTS
1386767,2024-12-10,1,ZTS
1386768,2024-12-11,1,ZTS
1386769,2024-12-12,0,ZTS


In [16]:
# Process train and test set
news_train_df = pd.read_csv("./Data/news_data_train2.csv")
news_test_df = pd.read_csv("./Data/news_data_test2.csv")

news_train_combined_df = pd.merge(
    news_train_df, ticker_senti_date,
    left_on=["date", "Ticker"],
    right_on=["Date", "Ticker"],
    how="left"
)
news_train_combined_df = news_train_combined_df.drop(["date", "Ticker"], axis=1)
news_train_combined_df.dropna(subset=["Date"], inplace=True)

news_test_combined_df = pd.merge(
    news_test_df, ticker_senti_date,
    left_on=["date", "Ticker"],
    right_on=["Date", "Ticker"],
    how="left"  # Use 'left' to keep all rows from df1
)
news_test_combined_df = news_test_combined_df.drop(["date", "Ticker"], axis=1)
news_test_combined_df.dropna(subset=["Date"], inplace=True)

In [17]:
# Format
train_df = news_train_combined_df.copy()
train_df["Date"] = pd.to_datetime(train_df["Date"])
train_df['sentiment_label'] = train_df['sentiment_label'].astype(int)

test_df = news_test_combined_df.copy()
test_df["Date"] = pd.to_datetime(test_df["Date"])
test_df['sentiment_label'] = test_df['sentiment_label'].astype(int)

train_df.columns = [i.lower() for i in train_df.columns]
test_df.columns = [i.lower() for i in test_df.columns]

print(train_df.shape)
print(test_df.shape)

(52401, 6)
(9929, 6)


In [18]:
# Further split for train and val set
val_df = train_df[train_df['date'] >= "2024-11-01"].reset_index(drop=True)
X_train_df = train_df[train_df['date'] < "2024-11-01"].reset_index(drop=True)

print(X_train_df.shape)
print(val_df.shape)

(43098, 6)
(9303, 6)


In [19]:
X_train_df.to_csv("./Data/train_labelled2.csv", index=False)
val_df.to_csv("./Data/val_labelled2.csv", index=False)
train_df.to_csv("./Data/train_val_labelled2.csv", index=False)

In [20]:
test_df

,title,source,topic,company name(s) - cleaned,date,sentiment_label
0,Fred Alger Management LLC Has $1.98 Million St...,https://news.google.com/rss/articles/CBMixAFBV...,NaN,NaN,2024-12-02,1
1,International Markets and Agilent (A): A Deep ...,https://news.google.com/rss/articles/CBMijgFBV...,NaN,NaN,2024-12-02,1
2,Agilent Technologies: Early Signs Of Turnaroun...,https://news.google.com/rss/articles/CBMiqwFBV...,NaN,NaN,2024-12-02,1
3,Erste Asset Management GmbH Buys Shares of 113...,https://news.google.com/rss/articles/CBMiywFBV...,NaN,NaN,2024-12-02,1
4,"Fmr LLC Purchases 493,947 Shares of Agilent Te...",https://news.google.com/rss/articles/CBMisAFBV...,NaN,NaN,2024-12-03,0
...,...,...,...,...,...,...
11590,"Geode Capital Management LLC Buys 185,364 Shar...",https://news.google.com/rss/articles/CBMiwAFBV...,NaN,NaN,2024-12-11,1
11591,Toronto Dominion Bank Grows Stock Position in ...,https://news.google.com/rss/articles/CBMiugFBV...,NaN,NaN,2024-12-12,0
11592,Stock Yards Bank & Trust Co. Has $17.39 Millio...,https://news.google.com/rss/articles/CBMixgFBV...,NaN,NaN,2024-12-12,0
11593,Zoetis Boosts Quarterly Dividend by 16% to Rec...,https://news.google.com/rss/articles/CBMitgFBV...,NaN,NaN,2024-12-12,0


In [21]:
test_df.to_csv("./Data/test_labelled2.csv", index=False)